# Predictive DWA Simulation Data Analysis

This notebook analyzes data from Predictive DWA simulations, including robot paths, performance metrics, and statistical analysis.

## Features:
- Robot path visualization from multiple simulations
- Performance metrics analysis (velocity, collisions, goal success)
- Statistical summaries and distributions
- Interactive plots and data exploration


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

# Set figure size for better visibility
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")


## Data Loading and Preprocessing


In [ ]:
# Configuration
DATA_DIR = "simulation_data"
CORRIDOR_WIDTH = 4.0  # Adjust based on your simulation parameters
CORRIDOR_LENGTH = 20.0  # Adjust based on your simulation parameters

# Check if data directory exists
if not os.path.exists(DATA_DIR):
    print(f"Data directory '{DATA_DIR}' not found. Please run simulations first.")
else:
    print(f"Data directory '{DATA_DIR}' found.")
    
    # List available CSV files
    csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
    print(f"Found {len(csv_files)} CSV files:")
    for file in sorted(csv_files)[:10]:  # Show first 10 files
        print(f"  - {os.path.basename(file)}")
    if len(csv_files) > 10:
        print(f"  ... and {len(csv_files) - 10} more files")


In [ ]:
def load_simulation_data(file_path):
    """Load and preprocess simulation data from CSV file"""
    try:
        df = pd.read_csv(file_path)
        
        # Extract simulation ID from filename
        filename = os.path.basename(file_path)
        if 'mc_run_' in filename:
            sim_id = filename.split('_')[2].split('.')[0]
        else:
            sim_id = filename.split('_')[-1].split('.')[0]
        
        df['simulation_id'] = sim_id
        df['filename'] = filename
        
        # Convert timestamp to datetime
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        
        # Calculate additional metrics
        df['speed'] = np.sqrt(df['robot_velocity_x']**2 + df['robot_velocity_y']**2)
        df['distance_from_start'] = np.sqrt((df['robot_x'] - df['robot_x'].iloc[0])**2 + 
                                          (df['robot_y'] - df['robot_y'].iloc[0])**2)
        
        return df
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

# Load all simulation data
all_simulations = []
simulation_summaries = []

for file_path in csv_files:
    if 'monte_carlo_summary' in file_path:
        # Load summary data
        summary_df = pd.read_csv(file_path)
        simulation_summaries.append(summary_df)
    else:
        # Load individual simulation data
        sim_data = load_simulation_data(file_path)
        if sim_data is not None:
            all_simulations.append(sim_data)

print(f"Loaded {len(all_simulations)} individual simulations")
print(f"Loaded {len(simulation_summaries)} summary files")

if all_simulations:
    # Combine all simulations
    combined_data = pd.concat(all_simulations, ignore_index=True)
    print(f"Combined data shape: {combined_data.shape}")
    print(f"Columns: {list(combined_data.columns)}")


## Robot Path Visualization

This section visualizes the robot paths from multiple simulations on the same plot.


In [ ]:
def plot_robot_paths(data, max_simulations=20, corridor_width=4.0, corridor_length=20.0):
    """Plot robot paths from multiple simulations"""
    
    # Get unique simulation IDs
    unique_sims = data['simulation_id'].unique()[:max_simulations]
    
    plt.figure(figsize=(15, 8))
    
    # Plot corridor boundaries
    plt.plot([0, corridor_length], [0, 0], 'k-', linewidth=3, label='Corridor boundaries')
    plt.plot([0, corridor_length], [corridor_width, corridor_width], 'k-', linewidth=3)
    plt.plot([0, 0], [0, corridor_width], 'k-', linewidth=3)
    plt.plot([corridor_length, corridor_length], [0, corridor_width], 'k-', linewidth=3)
    
    # Plot door (assuming right side door at 40% of corridor length)
    door_x = 0.4 * corridor_length
    plt.plot([door_x, door_x], [corridor_width-0.5, corridor_width+0.5], 'g-', linewidth=5, label='Door')
    
    # Plot robot paths
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_sims)))
    
    for i, sim_id in enumerate(unique_sims):
        sim_data = data[data['simulation_id'] == sim_id]
        
        # Plot path
        plt.plot(sim_data['robot_x'], sim_data['robot_y'], 
                color=colors[i], alpha=0.7, linewidth=1.5, 
                label=f'Sim {sim_id}' if i < 10 else None)
        
        # Mark start and end points
        plt.scatter(sim_data['robot_x'].iloc[0], sim_data['robot_y'].iloc[0], 
                  color=colors[i], s=50, marker='o', alpha=0.8)
        plt.scatter(sim_data['robot_x'].iloc[-1], sim_data['robot_y'].iloc[-1], 
                  color=colors[i], s=50, marker='s', alpha=0.8)
    
    # Mark goal position (assuming it's at the end of corridor)
    goal_x = corridor_length - 1.0
    goal_y = corridor_width / 1.2
    plt.scatter(goal_x, goal_y, color='red', s=100, marker='*', 
               label='Goal', zorder=10)
    
    plt.xlabel('X Position (meters)')
    plt.ylabel('Y Position (meters)')
    plt.title(f'Robot Paths from {len(unique_sims)} Simulations')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.axis('equal')
    plt.tight_layout()
    plt.show()

# Plot robot paths
if all_simulations:
    plot_robot_paths(combined_data, max_simulations=20)
